### 01 Introduction to using TOAST

(This is specifically written for use on SO-UK servers)

In [16]:
import toast
import os
import datetime
import numpy as np
from astropy import units as u

import numpy as np
from pathlib import Path
import sys

# Finds the root directory by walking up to a known anchor file
root_dir = Path("01_intro.ipynb").resolve().parents[1] 
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))


In [14]:
env = toast.utils.Environment.get()
log = toast.utils.Logger.get()
gt = toast.timing.GlobalTimers.get()
gt.start("toast_ground_sim (total)")
timer0 = toast.timing.Timer()
timer0.start()

# Get optional MPI parameters
comm, procs, rank = toast.get_world()

if "OMP_NUM_THREADS" in os.environ:
    nthread = os.environ["OMP_NUM_THREADS"]
else:
    nthread = "unknown number of"
log.info_rank(
    f"Executing workflow with {procs} MPI tasks, each with "
    f"{nthread} OpenMP threads at {datetime.datetime.now()}",
    comm,
)

TOAST INFO: Executing workflow with 1 MPI tasks, each with unknown number of OpenMP threads at 2026-08-12 20:31:43.704063


The operators we want to configure from the command line or a parameter file.
We will use other operators, but these are the ones that the user can configure.
The "name" of each operator instance controls what the commandline and config
file options will be called.

We can also set some default values here for the traits, including whether an
operator is disabled by default.

In [15]:
operators = [
    toast.ops.SimGround(name="sim_ground", weather="atacama", detset_key="pixel"),
    toast.ops.DefaultNoiseModel(name="default_model", noise_model="noise_model"),
    toast.ops.ElevationNoise(name="elevation_model", out_model="noise_model"),
    toast.ops.PointingDetectorSimple(name="det_pointing_azel", quats="quats_azel"),
    toast.ops.StokesWeights(
        name="weights_azel", weights="weights_azel", mode="IQU"
    ),
    toast.ops.PointingDetectorSimple(
        name="det_pointing_radec", quats="quats_radec"
    ),
    toast.ops.ScanHealpixMap(name="scan_healpix_map", enabled=False),
    toast.ops.ScanWCSMap(name="scan_wcs_map", enabled=False),
    toast.ops.SimAtmosphere(name="sim_atmosphere"),
    toast.ops.SimCatalog(name="sim_catalog", enabled=False),
    toast.ops.SimScanSynchronousSignal(name="sim_sss", enabled=False),
    toast.ops.TimeConstant(
        name="convolve_time_constant", deconvolve=False, enabled=False
    ),
    toast.ops.GainScrambler(name="gain_scrambler", enabled=False),
    toast.ops.SaveHDF5(name="save_hdf5", enabled=False),
    toast.ops.SimNoise(name="sim_noise"),
    toast.ops.PixelsHealpix(name="pixels_healpix_radec"),
    toast.ops.PixelsWCS(
        name="pixels_wcs_radec",
        projection="CAR",
        resolution=(0.005 * u.degree, 0.005 * u.degree),
        auto_bounds=True,
        enabled=False,
    ),
    toast.ops.PixelsWCS(
        name="pixels_wcs_azel",
        projection="CAR",
        resolution=(0.005 * u.degree, 0.005 * u.degree),
        auto_bounds=True,
        enabled=False,
    ),
    toast.ops.StokesWeights(name="weights_radec", mode="IQU"),
    toast.ops.YieldCut(name="yield_cut", enabled=False),
    toast.ops.ScanHealpixMask(name="processing_mask", enabled=False),
    toast.ops.FlagSSO(name="flag_sso", enabled=False),
    toast.ops.CadenceMap(name="cadence_map", enabled=False),
    toast.ops.CrossLinking(name="crosslinking", enabled=False),
    toast.ops.Statistics(name="raw_statistics", enabled=False),
    toast.ops.TimeConstant(
        name="deconvolve_time_constant", deconvolve=True, enabled=False
    ),
    toast.ops.HWPFilter(name="hwpfilter", enabled=False),
    toast.ops.GroundFilter(name="groundfilter", enabled=False),
    toast.ops.PolyFilter(name="polyfilter1D"),
    toast.ops.PolyFilter2D(name="polyfilter2D", enabled=False),
    toast.ops.CommonModeFilter(name="common_mode_filter", enabled=False),
    toast.ops.Statistics(name="filtered_statistics", enabled=False),
    toast.ops.BinMap(name="binner", pixel_dist="pix_dist"),
    toast.ops.MapMaker(name="mapmaker"),
    toast.ops.PixelsHealpix(name="pixels_healpix_radec_final", enabled=False),
    toast.ops.PixelsWCS(name="pixels_wcs_radec_final", enabled=False),
    toast.ops.PixelsWCS(name="pixels_wcs_azel_final", enabled=False),
    toast.ops.BinMap(
        name="binner_final", enabled=False, pixel_dist="pix_dist_final"
    ),
    toast.ops.FilterBin(
        name="filterbin",
        enabled=False,
    ),
    toast.ops.MemoryCounter(name="mem_count", enabled=False),
]